# Sales Data Cleaning and Analysis
This notebook documents the full cleaning process for a 4,215-record sales dataset (2024-2025) with the typical problems of a file exported without quality control: inconsistent categories, numeric columns stored as text, mixed date formats, duplicates, and impossible values.

The goal isn't just to make the data usable, but to leave a record of what was corrected and on what basis, so the results are auditable.

**Tools:** Python · pandas


## 1. Initial Diagnosis

Before changing anything, the file's condition is inspected: data types, null values, numeric ranges, and category consistency. Cleaning without diagnosing leads to fixing the obvious and missing what actually matters.


In [ ]:
import pandas as pd
df = pd.read_csv("ventas_sucio.csv")
df.head(20)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["region"].value_counts()

### Issues found

- **6 of 7 columns are read as text.** `ingreso` (revenue) and `unidades_vendidas` (units sold)
  contain currency symbols, decimal commas, and placeholder values for missing
  data ("N/A", "-"), which blocks any calculation until they're converted.
- **The `region` column has 28 distinct values** when the business
  operates in 5 regions. Variants from capitalization and spacing are counted
  separately and fragment the totals.
- **53 records with no region reported**, encoded as "?" or
  "unknown" instead of null.
- **Prices out of range:** minimum of €0 and maximum of €21,631 against a
  median of €52. The standard deviation (671) is more than six times the mean,
  a sign of extreme values distorting any average.
- **Inconsistent column names**, with trailing spaces and mixed
  capitalization.
- **Duplicate rows and completely empty rows.**


## 2. Normalizing Column Names

Headers arrive with trailing spaces and no consistent capitalization
(`"Producto "`, `"Precio Unitario"`, `"region"`). A trailing space is invisible
when reading the file but causes errors when referencing the column.

They're unified to lowercase, no spaces, separated by underscores. This is the
first step because every later operation depends on being able to reference
columns predictably.


In [ ]:
df.columns = (df.columns
              .str.strip()
              .str.lower()
              .str.replace(" ", "_"))

df.columns

## 3. Removing Empty Rows and Duplicates

Rows with no data at all (a common export artifact) and identical records,
which would artificially inflate sales totals, are removed.

`dropna(how="all")` is used to delete only fully empty rows: the default
behavior would remove any row with a single null and discard hundreds of
valid records.

The row count is logged before and after to document the volume removed.


In [ ]:
print("Before:", len(df))

df = df.dropna(how="all")
df = df.drop_duplicates()

print("After:", len(df))


## 4. Unifying Categories

`region`, `canal`, and `producto` contain the same value written in different
forms. To pandas, "Norte", "NORTE", and "  norte  " are three distinct
categories: totals get split up and charts show duplicate bars.

Trailing spaces are stripped and capitalization is normalized. Values
representing missing data ("?", "unknown") are converted to null, since they
aren't a real category but unrecorded information.


In [ ]:
# Normalize format: trim whitespace and capitalize
for col in ["region", "canal", "producto"]:
    df[col] = df[col].str.strip().str.capitalize()

# Placeholder values aren't a real category: convert to null
df["region"] = df["region"].replace(["?", "Desconocida", ""], pd.NA)

df["region"].value_counts(dropna=False)


In [ ]:
df["canal"].value_counts()

In [ ]:
equivalencias_canal = {
    "On-line": "Online",
    "Tienda fisica": "Tienda",
    "Tienda física": "Tienda",
    "Distrib.": "Distribuidor",
}

df["canal"] = df["canal"].replace(equivalencias_canal)
df["canal"].value_counts()

Format normalization resolves capitalization and spacing variants, but not
naming differences ("Distrib." vs. "Distribuidor", "Tienda física" vs.
"Tienda"). These require an explicit equivalence mapping: no automatic
transformation can infer that two different labels refer to the same
category.


## 5. Converting Numeric Columns

`ingreso` (revenue) and `unidades_vendidas` (units sold) are stored as text,
which blocks any calculation: summing a text column concatenates values
instead of adding them. It's the most common problem in exported files, and a
single non-numeric value is enough to render the whole column unusable.

**Revenue.** Contains the currency symbol in a variable position (before or
after), a comma as the decimal separator, and a period as the thousands
separator, following Spanish convention. The cleaning sequence follows a
mandatory order: the thousands separator is removed first, and only then is
the decimal comma replaced with a period. Reversing these two steps would
produce values with two periods —not interpretable as a number— and the
conversion would silently return nulls.

**Units sold.** Placeholder values for missing data ("N/A", "-", "null") can't
be converted to numbers. `errors="coerce"` is used to turn them into nulls
instead of stopping execution: they're unrecorded data and should be treated
as such, not as zeros.

The resulting data type is checked in the same cell, and the nulls generated
are counted, since any conversion with `coerce` carries a potential loss of
information that should be documented.


In [ ]:
# Revenue: remove currency symbol and normalize decimal separators
df["ingreso"] = (df["ingreso"]
                 .astype(str)
                 .str.replace("€", "", regex=False)
                 .str.strip()
                 .str.replace(".", "", regex=False)
                 .str.replace(",", ".", regex=False))

df["ingreso"] = pd.to_numeric(df["ingreso"], errors="coerce")

# Units: placeholder values ("N/A", "-", "null") become null
df["unidades_vendidas"] = pd.to_numeric(df["unidades_vendidas"], errors="coerce")

df.dtypes

In [ ]:
df[["unidades_vendidas", "ingreso", "precio_unitario"]].isnull().sum()

The conversion doesn't cause any loss of information: `ingreso` converts fully
with no nulls, and the missing values in `unidades_vendidas` (121) and
`precio_unitario` (96) were already present in the source file. The resulting
nulls reflect unrecorded data, not errors introduced during cleaning.


In [ ]:
df.describe().T

## 6. Handling Impossible Values

The diagnosis reveals values that can't correspond to real transactions:
negative quantities, quantities three orders of magnitude above the median,
and prices of €0 or above €21,000 in a catalog with a €53 median.

**Identification criterion.** These aren't legitimate outliers but capture
errors, and cross-checking confirms it: records with disproportionate
quantities show revenue within the normal range. A sale of 11,904 units would
have generated hundreds of thousands of euros in revenue; the maximum in the
dataset is €1,190. Both figures can't be true at the same time.

**Treatment criterion.** Affected values are converted to null rather than
deleting the whole record: the rest of the fields in those rows (date,
product, region, channel) are valid, and removing them would distort the
category counts.

**Reconstruction.** Since `ingreso` was preserved intact and follows the
relationship `units × price`, the nulled values are recovered by solving for
the unknown from the two remaining fields. This redundancy makes it possible
to restore information instead of discarding it.

**Thresholds applied.** Set at 100 units and €1,000, since they sit well above
observed behavior and well below the anomalous values. In a real engagement
these limits should be validated with the client, since legitimate wholesale
orders would change the criteria.


In [ ]:
import numpy as np

# Units: no negative sales or out-of-scale quantities
df.loc[df["unidades_vendidas"] < 1, "unidades_vendidas"] = np.nan
df.loc[df["unidades_vendidas"] > 100, "unidades_vendidas"] = np.nan

# Price: not free, not outside the catalog range
df.loc[df["precio_unitario"] <= 0, "precio_unitario"] = np.nan
df.loc[df["precio_unitario"] > 1000, "precio_unitario"] = np.nan

df.describe().T

In [ ]:
# Recover units from revenue and price
faltan_unidades = df["unidades_vendidas"].isna() & df["precio_unitario"].notna()
df.loc[faltan_unidades, "unidades_vendidas"] = (
    df.loc[faltan_unidades, "ingreso"] / df.loc[faltan_unidades, "precio_unitario"]
).round()

# Recover price from revenue and units
faltan_precio = df["precio_unitario"].isna() & df["unidades_vendidas"].notna()
df.loc[faltan_precio, "precio_unitario"] = (
    df.loc[faltan_precio, "ingreso"] / df.loc[faltan_precio, "unidades_vendidas"]
).round(2)

df.describe().T

In [ ]:
df[df["precio_unitario"] > 400]

In [ ]:
df.groupby("producto")["precio_unitario"].describe()

### Limitation of Redundancy-Based Reconstruction

Verification detects prices of €839 and €439 for the "Mouse" product, whose
median sits at €24.75. Both come from the reconstruction process: their price
was derived from a `revenue` value that was itself wrong.

This exposes the limit of the method: reconstructing one field from others is
only reliable if the source field is correct.

**Fix applied.** A global threshold isn't enough when products have very
different price ranges: €839 is impossible for a mouse but plausible for a
monitor. It's replaced with a threshold relative to each product's median,
which nulls 3 records —one more than those identified through visual
inspection— and reduces the "Mouse" product's standard deviation from 27.49 to
1.84, in line with the rest of the catalog.


In [ ]:
# Relative threshold: no price should be more than 3x its product's median
mediana_producto = df.groupby("producto")["precio_unitario"].transform("median")
df.loc[df["precio_unitario"] > mediana_producto * 3, "precio_unitario"] = np.nan
df.groupby("producto")["precio_unitario"].describe()

In [ ]:
precio_canal = df.groupby(["producto", "canal"])["precio_unitario"].mean().unstack()
precio_canal["descuento_distribuidor_%"] = (
    (precio_canal["Distribuidor"] / precio_canal[["Online", "Tienda"]].mean(axis=1) - 1) * 100
).round(1)
precio_canal.round(2)

## 7. Normalizing Dates

The column mixes three different formats (ISO, European, and US), the result
of an export with no unified standard. `format="mixed"` is used to interpret
each record individually, and `dayfirst=True` to resolve ambiguous dates
following European convention.

This last parameter is decisive: without it, a date like 05/12/2024 would be
read as May 5th instead of December 5th. The error produces no warning and
only shows up when data is aggregated by month, distorting the entire
seasonality analysis.

**Verification.** The resulting range spans from January 1, 2024 to December
31, 2025, consistent with the expected period. The absence of dates outside
that interval confirms no format was misread. 60 null values (1.5%) are
recorded, corresponding to missing dates in the source.


In [ ]:
df["fecha"] = pd.to_datetime(df["fecha"], format="mixed", dayfirst=True, errors="coerce")
df["fecha"].dtype


In [ ]:
df["fecha"].isna().sum()

In [ ]:
print(df["fecha"].min(), "→", df["fecha"].max())

## 8. Pricing Policy by Channel

Each product's average price varies systematically depending on the sales
channel. The Distributor channel applies a discount of between 17.7% and
18.2% relative to retail channels, with a variation of less than half a
percentage point across the catalog's five products.

**Interpretation.** The uniformity of the discount points to an established
wholesale policy rather than case-by-case negotiated deals: a differentiated
rate applied consistently. The Online and In-store channels keep nearly
identical prices between themselves, suggesting the company doesn't
differentiate price between its own channels and only separates wholesale
from retail.

**Implication.** A revenue-by-channel analysis that ignores this difference
would attribute lower commercial performance to the Distributor channel when
part of the gap is explained by pricing structure, not sales volume. The
relevant business question is whether the extra volume this channel brings
offsets the margin given up.


In [ ]:
resumen_canal = df.groupby("canal").agg(
    operaciones=("ingreso", "count"),
    unidades=("unidades_vendidas", "sum"),
    ingreso_total=("ingreso", "sum"),
    ticket_medio=("ingreso", "mean")
).round(2)

resumen_canal

## 9. Profitability of the Distributor Channel

The Distributor channel contributes €142,112 (15% of total revenue), compared
to €471,770 from the Online channel and €326,096 from In-store.

**Volume doesn't justify the discount applied.** The number of units per
transaction is nearly identical across all three channels (3.03 for
Distributor vs. 3.01 for Online and In-store). The wholesale channel doesn't
buy larger quantities per transaction: it buys the same quantity at an 18%
lower price.

The difference in average ticket —€197.10 vs. €236.00 and €244.27— corresponds
almost exactly to the discount applied, which rules out a different buying
pattern as the explanation.

**Recommendation.** Review the terms of the Distributor channel. The wholesale
discount is being applied without the volume trade-off that would justify it.
It's worth assessing whether it brings value through other means not captured
in this data —geographic reach, customer acquisition cost, recurrence— before
changing the policy. Based on the data available, it's the lowest-performing
channel per transaction.


In [ ]:
resumen_canal["unidades_por_operacion"] = (
    resumen_canal["unidades"] / resumen_canal["operaciones"]
).round(2)
resumen_canal

In [ ]:
# Export the cleaned dataset
df.to_csv("ventas_limpio.csv", index=False)

print(f"Final records: {len(df)}")
print(f"Columns: {list(df.columns)}")

## 10. Visualizing Results

### Effect of Cleaning on the Quantity Distribution

Comparing the original and cleaned datasets shows the impact of the anomalous
values. In the original, the axis extends to 11,904 units because of a
handful of erroneous records, concentrating 99% of observations into a single
unreadable bar. After cleaning, the real distribution falls between 1 and 5
units.

This is a check worth running always: a histogram whose axis extends far
beyond the mass of the data indicates the presence of extreme values, even if
measures of central tendency don't show it.

**Observation.** The five quantities show nearly identical frequencies (around
800 transactions each). In a real sales dataset, a decreasing distribution
would be expected, with single-unit purchases dominating. This uniformity is
consistent with the dataset's simulated origin and should be kept in mind when
interpreting the results.


In [ ]:
df_original = pd.read_csv("ventas_sucio.csv")

In [ ]:
import matplotlib.pyplot as plt

# Units from the original dataset, converted to numeric but not cleaned
unidades_sucias = pd.to_numeric(df_original["unidades_vendidas"], errors="coerce")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before
axes[0].hist(unidades_sucias.dropna(), bins=50, color="#c44e52")
axes[0].set_title("Before: mean 40.3 units")
axes[0].set_xlabel("Units per transaction")
axes[0].annotate("max: 11,904 units", xy=(11904, 100), xytext=(6000, 2000),
                 arrowprops=dict(arrowstyle="->"), fontsize=9)

# After
axes[1].hist(df["unidades_vendidas"].dropna(),
             bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
             color="#55a868", edgecolor="white")
axes[1].set_title("After: mean 3.0 units")
axes[1].set_xlabel("Units per transaction")
axes[1].set_xticks([1, 2, 3, 4, 5])

plt.tight_layout()
plt.show()

---


In [ ]:
precio_canal[["Distribuidor", "Online", "Tienda"]].plot(
    kind="bar",
    figsize=(9, 4.5),
    color=["#c44e52", "#4c72b0", "#55a868"]
)

plt.title("Average price by product and channel")
plt.ylabel("Average price (€)")
plt.xlabel("")
plt.xticks(rotation=0)
plt.legend(title="Channel")
plt.tight_layout()
plt.show()

In [ ]:
descuentos = precio_canal["descuento_distribuidor_%"].sort_values()

ax = descuentos.plot(kind="barh", figsize=(8, 3.5), color="#c44e52")
ax.set_title("Distributor channel discount vs. retail price")
ax.set_xlabel("Difference (%)")
ax.set_ylabel("")
ax.axvline(descuentos.mean(), color="black", linestyle="--", linewidth=1)
for i, v in enumerate(descuentos):
    ax.text(v + 0.4, i, f"{v}%", va="center", ha="left",
            fontsize=9, color="white", fontweight="bold")    
plt.tight_layout()
plt.show()

### Pricing Structure by Channel

The first chart shows each product's average price by channel. The
Distributor channel's difference is visible across all five products, though
the absolute scale masks it in lower-priced items: the discount on the mouse
is the largest in the catalog in relative terms and the least noticeable in
the chart.

The second chart corrects that distortion by showing the difference in
percentage terms. All five bars converge around the average (dashed line)
with a spread of less than half a point, confirming a uniformly applied
wholesale rate rather than individually negotiated deals.

Both representations are necessary: the first shows the scale of the
business, the second reveals the pattern. Either one alone would lead to an
incomplete reading.


---


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Average ticket: there is a difference
resumen_canal["ticket_medio"].plot(kind="bar", ax=axes[0], color="#4c72b0")
axes[0].set_title("Average ticket per transaction (€)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)

# Units per transaction: no difference
resumen_canal["unidades_por_operacion"].plot(kind="bar", ax=axes[1], color="#937860")
axes[1].set_title("Units per transaction")
axes[1].set_xlabel("")
axes[1].set_ylim(0, 4)
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### Average Ticket vs. Volume per Transaction

Comparing both metrics answers the question raised by the pricing structure:
if the Distributor channel gets an 18% discount, you'd expect it to buy
larger quantities per transaction.

The left-hand chart shows an average ticket of €197 for Distributor vs. €236
and €244 for retail channels. The right-hand chart shows units per
transaction are equivalent across all three channels (3.03 vs. 3.01). The
ticket difference, therefore, doesn't reflect a different buying pattern —it
comes exclusively from the price applied.

**Note on scale.** The right-hand chart's axis is deliberately fixed between 0
and 4. With an automatic scale, a 0.02-unit difference would produce wildly
different bar heights and suggest a disparity that doesn't exist. Starting
the axis at zero represents the true magnitude of the difference.


## 11. Conclusions

**On data quality.** The original file was unusable for analysis: six of its
seven columns were read as text, and the quantity column showed an average of
40.27 units per transaction against a real value of 3.02. Any report built
without prior cleaning would have reported a figure off by more than 1,200%.

**On the process.** 4,215 records were cleaned down to 4,055 valid ones, with
a 3.8% loss attributable to duplicates and empty rows. Impossible values were
nulled rather than deleted, to avoid discarding the correct fields in those
same records, and were recovered using the relationship
`revenue = units × price` wherever the remaining fields allowed it.

**On the business.** The Distributor channel operates with a flat 18%
discount that isn't justified by transaction volume: it buys 3.03 units per
transaction vs. 3.01 in retail channels. It's recommended to review this
channel's terms before renewing them.

**Limitations.** The 60 records with no date (1.5%) and 72 with no region
(1.8%) are kept as nulls and excluded from the time-based and geographic
analyses, respectively. The anomalous-price detection threshold was set at
three times each product's median; in a real engagement it should be
validated with the client.

---

*Cleaned dataset exported as `ventas_limpio.csv`.*
